## **Feature Engineering**

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load the Data from the data creation script

In [12]:
# 1. Search for the specific file starting from the parent directory
file_generator = Path.cwd().parent.rglob("transactions_2026.csv")

# 2. Get the path of the first match
target_file = next(file_generator)

# 3. Read it directly into Pandas
df = pd.read_csv(target_file)

## Feature engineering

In [13]:
# Convert 'timestamp' to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Ensure correct ordering - critical for time-based features
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

### Define functions to generate new features

In [14]:
import numpy as np
import pandas as pd


# =======================================================================================
# Haversine distance (km)
# =======================================================================================
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth.
    Inputs are in degrees. Output is in kilometers.
    """
    R = 6371.0  # Earth radius in km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# =======================================================================================
# Feature engineering
# =======================================================================================
def engineer_features(df):

    # =================================================================================
    # 0. Copy dataframe
    # =================================================================================
    
    # Ensure we don't modify the original dataframe(raw data)
    df = df.copy()

    
    
    # =================================================================================
    # 1. Time-based features
    # =================================================================================
    
    # Extract hour of day and day of week
    
    # Because fraud patterns may vary by time windows 
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek

    
    
    # =================================================================================
    # 2. Transaction velocity
    # =================================================================================
    
    
    # Number of transactions in the past 24 hours per user
    
    # High velocity (many transactions in a short period) 
    # is a primary indicator of account takeover or automated fraud.  
    
    # First we calculate it separately to ensure we don't get into alignment issues
    counts = (
        df
        .groupby('user_id')
        .rolling('24h', on='timestamp')['tx_id']
        .count()
        .values
    )
    
    # Then assingn it back to a column
    df['tx_count_24h'] = counts

    
    
    # =================================================================================
    # 3. Behavioral features
    # =================================================================================
       
    # Calculate average spend per user up to (but not including) current transaction
    df['avg_spend_user'] = (
        df
        .groupby('user_id')['amount']
        .transform(lambda x: x.shift(1).expanding().mean())
    )
    
    # amount to average spenditure ratio - handle division by zero
    
    # Because a ratio significantly higher than 1 (e.g., spending 10x the usual amount) 
    # is a strong fraud signal.
    df['amount_ratio'] = np.where(
        df['avg_spend_user'] > 0, 
        df['amount'] / df['avg_spend_user'], 
        0
    )

    
    
    # =================================================================================
    # 4. Geospatial features
    # =================================================================================
    
    # Previous transaction coordinates and timestamp
    df['prev_lat'] = df.groupby('user_id')['lat'].shift(1)
    df['prev_lon'] = df.groupby('user_id')['lon'].shift(1)
    df['prev_ts']  = df.groupby('user_id')['timestamp'].shift(1)

    # Distance from previous transaction (km)
    df['dist_from_last_tx_km'] = haversine(
        df['lat'], 
        df['lon'],
        df['prev_lat'], 
        df['prev_lon']
    ).fillna(0)

    # Calculate time difference in hours
    time_diff_hours = (
        (df['timestamp'] - df['prev_ts'])
        .dt.total_seconds()
        .div(3600)
        .clip(lower=1e-3)
    )

    # Calculate travel velocity (km/h) - handle division by zero
    
    # If a user transacts in New York and then 10 minutes later 
    # in London, the high velocity flags a likely fraud event.
    df['travel_velocity_kmph'] = np.where(
        time_diff_hours == 0,
        df['dist_from_last_tx_km'] / time_diff_hours,
        0
    )



    # =================================================================================
    # 5. Categorical encoding
    # =================================================================================
    df = pd.get_dummies(
        df,
        columns=['auth_method', 'category'],
        drop_first=True
    )

    
    
    # =================================================================================
    # 6. Drop non-model columns
    # =================================================================================
    
    df = df.drop(
        columns=[
            'tx_id',
            'prev_lat',
            'prev_lon',
            'prev_ts'
        ]
    )

    
    
    # =================================================================================
    # 7. Final numeric safety
    # =================================================================================
    num_cols = df.select_dtypes(include='number').columns
    df[num_cols] = df[num_cols].fillna(0)

    return df

### Exporting the data with new features

In [16]:
# Run feature engineering
df_final = engineer_features(df)

# Save output
df_final.to_csv("./artifacts/data/fraud_features_ready.csv", index=False)

print("Features Engineered!")
print("Final shape:", df_final.shape)
print("Number of NaNs left:", df_final.isna().sum().sum())

OSError: Cannot save file into a non-existent directory: 'artifacts\data'